# Inspect the Luna ICL checkpoint

This notebook reads saved prompts, demonstrations and results. It makes no API calls. The checkpoint contains 26 successful calibrated examples from 30 candidate targets on 10 training layouts; validation uses 30 states on 10 different layouts.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

REPO_PATH: str | None = None  # Set an absolute checkout path if needed.
_candidates = [Path(REPO_PATH).expanduser()] if REPO_PATH else [
    candidate for parent in (Path.cwd(), *Path.cwd().parents)
    for candidate in (parent, parent / "CaRL")
]
ROOT = next((path for path in _candidates if (path / "carl/icl").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Set REPO_PATH to the CaRL checkout containing carl/icl.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from carl.icl.boards import from_rows  # noqa: E402 -- checkout path is configured above.
from carl.icl.checkpoints.types import GeneratorCheckpoint  # noqa: E402
from carl.icl.evaluation.board_graphics import board_comparison  # noqa: E402

BASE = ROOT / "analysis/luna_icl/campaign_100/generator_probe_v2"
CHECKPOINT_DIR = BASE / "checkpoint_balanced"
checkpoint = GeneratorCheckpoint.model_validate(json.loads((CHECKPOINT_DIR / "generator.json").read_text()))
print(f"{checkpoint.model} | {checkpoint.actual_example_count} demonstrations | L={checkpoint.action_cap}")
print(f"Checkpoint SHA-256: {checkpoint.sha256}")
print(f"Frozen CLLP: {checkpoint.policy_fingerprint}")
display(pd.DataFrame([{
    "index": index, "trajectory": row.context, "target": row.target_index,
    "k": row.k, "hits": row.hits, "trials": row.trials, "d": row.d,
    "weight": row.weight, "u": row.u, "median_d": row.a_c,
} for index, row in enumerate(checkpoint.examples)]))

## One exact demonstration

Change `DEMO_INDEX` to inspect another source/target pair. `k` is its offline trajectory offset; `d` is the measured mean successful first-hit time of the frozen controller. They need not be equal.

In [ ]:
DEMO_INDEX = 0
example = checkpoint.examples[DEMO_INDEX]
print(f"{example.context}: k={example.k}, d={example.d:g}, u={example.u:g}, hits={example.hits}/{example.trials}")
board_comparison(from_rows(list(example.state)), from_rows(list(example.subgoal)), ("Training source", "Training subgoal")).show()

## Exact prompt and few-shot messages

The saved bundle includes complete generator requests for 4 and 26 examples, plus the policy and value prompts. No inference is performed when selecting or displaying a request.

In [ ]:
bundle = json.loads((CHECKPOINT_DIR / "generator_inputs.json").read_text())
REQUEST_INDEX = 0  # 0..3: four/all demonstrations at u=0.5/0.85
request = bundle["generator"][REQUEST_INDEX]
count, percentile = request["example_count"], request["u"]
print(f"examples={count}, u={percentile}")
for index, message in enumerate(request["payload"]["input"]):
    role = message["role"]
    display(Markdown(f"### Message {index}: {role}"))
    print(message["content"][0]["text"])

## Saved validation results

The original paired comparison reached 23/30 endpoints for both context sizes. A post-hoc second generator attempt recovered all seven rejected 26-shot queries, yielding 30/30 with replayed first replies. That is not an independent retest or 30 solved boards. Inspect source-level CSVs and the separate full-solver result below.

In [ ]:
queries = pd.read_csv(BASE / "report/queries.csv")
display(queries[["experiment", "variant", "board_id", "source_id", "u", "hit", "first_hit", "invalid", "copies", "actions"]])
solver_path = BASE / "solver_validation/case.json"
if solver_path.exists():
    solver = json.loads(solver_path.read_text())
    display(pd.DataFrame([{key: value for key, value in solver["outcome"].items() if key not in {"events", "action_path", "subgoal_path"}}]))
    display(pd.DataFrame([solver["tree_summary"]]))
    tree_path = BASE / "solver_validation/search_tree.html"
    display(Markdown(f"[Full search tree]({tree_path})"))
else:
    print("No completed full-solver record is available yet.")